In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from sklearn.metrics import roc_curve
from itertools import combinations
import pandas as pd
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [2]:
DEVICE      = "cpu"
NUM_CLASSES = 40
IMG_HEIGHT  = 112
IMG_WIDTH   = 112

ORL_FULL_PATH = "../../dataset/ORL_limpio/Training"
CLAHE_FULL_X2 = "../../dataset/Tecnica De mejora de imagen/ORL_CLAHE_x2/Training"
CLAHE_FULL_X4 = "../../dataset/Tecnica De mejora de imagen/ORL_CLAHE_x4/Training"
SR_FULL_X2    = "../../dataset/Tecnica De mejora de imagen/ORL_SuperResolution_x2/Training"
SR_FULL_X4    = "../../dataset/Tecnica De mejora de imagen/ORL_SuperResolution_x4/Training"


In [3]:
BASE = "../../Semana 10/GhostFaceNetv2"

# Pesos ORL (modelo base)
ghost_orl = f"{BASE}/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_orl.pth"

# Pesos CLAHE x2
ghost_clahe_x2_ce      = f"{BASE}/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_clahe_x2_experimento2.pth"
ghost_clahe_x2_arcface = f"{BASE}/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_arcface_clahe_x2_experimento3.pth"
ghost_clahe_x2_nodfc   = f"{BASE}/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_noDFC_clahe_x2_experimento4.pth"

# Pesos CLAHE x4
ghost_clahe_x4_ce      = f"{BASE}/GhostFacenetV2_Clahe_x4/best_ghostfacenet_clahe_x4_experimento2.pth"
ghost_clahe_x4_arcface = f"{BASE}/GhostFacenetV2_Clahe_x4/best_ghostfacenet_arcface_clahe_x4_experimento3.pth"
ghost_clahe_x4_nodfc   = f"{BASE}/GhostFacenetV2_Clahe_x4/best_ghostfacenet_noDFC_clahe_x4_experimento4.pth"

# Pesos SR x2
ghost_sr_x2_ce         = f"{BASE}/GhostFaceNetV2_SR_x2/best_ghostfacenet_SR_x2_experimento2.pth"
ghost_sr_x2_arcface    = f"{BASE}/GhostFaceNetV2_SR_x2/best_ghostfacenet_arcface_SR_x2_experimento3.pth"
ghost_sr_x2_nodfc      = f"{BASE}/GhostFaceNetV2_SR_x2/best_ghostfacenet_noDFC_SR_x2_experimento4.pth"

# Pesos SR x4
ghost_sr_x4_ce         = f"{BASE}/GhostFaceNetV2_SR_x4/best_ghostfacenet_SR_x4_experimento2.pth"
ghost_sr_x4_arcface    = f"{BASE}/GhostFaceNetV2_SR_x4/best_ghostfacenet_arcface_SR_x4_experimento3.pth"
ghost_sr_x4_nodfc      = f"{BASE}/GhostFaceNetV2_SR_x4/best_ghostfacenet_noDFC_SR_x4_experimento4.pth"

# Pesos Lite-ESRGAN x2
ghost_le_x2_ce         = f"{BASE}/GhostFaceNetV2_LiteERSGAN_x2/best_ghostfacenet_LE_x2_experimento2.pth"
ghost_le_x2_arcface    = f"{BASE}/GhostFaceNetV2_LiteERSGAN_x2/best_ghostfacenet_arcface_LE_x2_experimento3.pth"
ghost_le_x2_nodfc      = f"{BASE}/GhostFaceNetV2_LiteERSGAN_x2/best_ghostfacenet_noDFC_LE_x2_experimento4.pth"

# Pesos Lite-ESRGAN x4
ghost_le_x4_ce         = f"{BASE}/GhostFaceNetV2_LiteERSGA_x4/best_ghostfacenet_LE_x4_experimento2.pth"
ghost_le_x4_arcface    = f"{BASE}/GhostFaceNetV2_LiteERSGA_x4/best_ghostfacenet_arcface_LE_x4_experimento3.pth"
ghost_le_x4_nodfc      = f"{BASE}/GhostFaceNetV2_LiteERSGA_x4/best_ghostfacenet_noDFC_LE_x4_experimento4.pth"

# Definición de la arquitectura

In [4]:
full_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def cargar_dataset_completo(path):
    dataset = datasets.ImageFolder(path, transform=full_transform)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    imagenes, etiquetas = [], []
    for img, label in loader:
        imagenes.append(img.squeeze(0))
        etiquetas.append(label.item())
    return imagenes, etiquetas, dataset.classes


In [5]:

# ---------------------------------------------------------------------------
# Bloques del modelo — GhostFaceNetV2
# ---------------------------------------------------------------------------
class ConvBNPReLU(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride=stride,
                               padding=padding, groups=groups, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = nn.PReLU(out_c)
 
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))
 
 
def _make_divisible(v, divisor=4, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v
 
 
class SqueezeExcite(nn.Module):
    """SE con PReLU en la reduccion."""
    def __init__(self, in_chs, se_ratio=0.25):
        super().__init__()
        reduced_chs = _make_divisible(int(in_chs * se_ratio))
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_reduce = nn.Conv2d(in_chs, reduced_chs, 1, bias=True)
        self.act = nn.PReLU(reduced_chs)
        self.conv_expand = nn.Conv2d(reduced_chs, in_chs, 1, bias=True)
 
    def forward(self, x):
        x_se = self.avg_pool(x)
        x_se = self.act(self.conv_reduce(x_se))
        x_se = self.conv_expand(x_se)
        return x * torch.sigmoid(x_se)
 
 
class DFCAttention(nn.Module):
    def __init__(self, chs, kernel_size=5):
        super().__init__()
        pad = kernel_size // 2
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv_h = nn.Conv2d(chs, chs, kernel_size=(1, kernel_size),
                                 padding=(0, pad), groups=chs, bias=False)
        self.conv_v = nn.Conv2d(chs, chs, kernel_size=(kernel_size, 1),
                                 padding=(pad, 0), groups=chs, bias=False)
        self.bn = nn.BatchNorm2d(chs)
 
    def forward(self, x):
        h, w = x.shape[-2:]
        att = self.pool(x)
        att = self.conv_h(att)
        att = self.conv_v(att)
        att = self.bn(att)
        att = torch.sigmoid(att)
        att = F.interpolate(att, size=(h, w), mode="nearest")
        return att

In [7]:
class GhostModule(nn.Module):
    def __init__(self, in_chs, out_chs, kernel_size=1, ratio=2, dw_size=3,
                 stride=1, use_act=True):
        super().__init__()
        self.out_chs = out_chs
        init_chs = math.ceil(out_chs / ratio)
        new_chs = init_chs * (ratio - 1)
 
        self.primary_conv = nn.Sequential(
            nn.Conv2d(in_chs, init_chs, kernel_size, stride,
                      kernel_size // 2, bias=False),
            nn.BatchNorm2d(init_chs),
            nn.PReLU(init_chs) if use_act else nn.Identity(),
        )
        self.cheap_operation = nn.Sequential(
            nn.Conv2d(init_chs, new_chs, dw_size, 1, dw_size // 2,
                      groups=init_chs, bias=False),
            nn.BatchNorm2d(new_chs),
            nn.PReLU(new_chs) if use_act else nn.Identity(),
        )
 
    def forward(self, x):
        x1 = self.primary_conv(x)
        x2 = self.cheap_operation(x1)
        out = torch.cat([x1, x2], dim=1)
        return out[:, :self.out_chs, :, :]
 
 
class GhostBottleneckV2(nn.Module):
    def __init__(self, in_chs, mid_chs, out_chs, dw_kernel_size=3,
                 stride=1, se_ratio=0.0, use_dfc=True):
        super().__init__()
        has_se = se_ratio is not None and se_ratio > 0.0
        self.stride = stride
        self.use_dfc = use_dfc

        if use_dfc:
            self.dfc = DFCAttention(mid_chs)          # <-- antes: DFCAttention(in_chs)

        self.ghost1 = GhostModule(in_chs, mid_chs, use_act=True)

        if stride > 1:
            self.conv_dw = nn.Conv2d(mid_chs, mid_chs, dw_kernel_size,
                                      stride=stride,
                                      padding=(dw_kernel_size - 1) // 2,
                                      groups=mid_chs, bias=False)
            self.bn_dw = nn.BatchNorm2d(mid_chs)
        else:
            self.conv_dw = None

        self.se = SqueezeExcite(mid_chs, se_ratio) if has_se else None

        self.ghost2 = GhostModule(mid_chs, out_chs, use_act=False)

        if in_chs == out_chs and stride == 1:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_chs, in_chs, dw_kernel_size, stride=stride,
                          padding=(dw_kernel_size - 1) // 2,
                          groups=in_chs, bias=False),
                nn.BatchNorm2d(in_chs),
                nn.Conv2d(in_chs, out_chs, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_chs),
            )
        self.out_channels = out_chs

    def forward(self, x):
        residual = x
        x1 = self.ghost1(x)

        if self.conv_dw is not None:
            x1 = self.bn_dw(self.conv_dw(x1))

        if self.use_dfc:
            att = self.dfc(x1)                      
            if att.shape[-2:] != x1.shape[-2:]:
                att = F.interpolate(att, size=x1.shape[-2:], mode="nearest")
            x1 = x1 * att

        if self.se is not None:
            x1 = self.se(x1)

        x1 = self.ghost2(x1)
        return x1 + self.shortcut(residual)

In [8]:
# ---------------------------------------------------------------------------
# Modelo — GhostFaceNetV2
# ---------------------------------------------------------------------------
class GhostFaceNetV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embedding_dim=128,
                 dropout_p=0.4, use_dfc=True):
        super().__init__()
 
        # Stem: conv 3x3 stride=2 -> 112x112 -> 56x56
        self.stem = ConvBNPReLU(1, 16, 3, stride=2, padding=1)
 
        # Etapa 1: 56x56 -> 28x28
        self.stage1 = nn.Sequential(
            GhostBottleneckV2(16, mid_chs=48, out_chs=24, dw_kernel_size=3,
                               stride=2, se_ratio=0.0, use_dfc=use_dfc)
        )
        c1 = self.stage1[-1].out_channels  # 24
 
        # Etapa 2: 28x28 -> 14x14 (1 stride=2 + 2 stride=1)
        stage2_blocks = [
            GhostBottleneckV2(c1, mid_chs=72, out_chs=40, dw_kernel_size=5,
                               stride=2, se_ratio=0.25, use_dfc=use_dfc)
        ]
        c2 = stage2_blocks[-1].out_channels
        for _ in range(2):
            stage2_blocks.append(
                GhostBottleneckV2(c2, mid_chs=120, out_chs=40, dw_kernel_size=5,
                                   stride=1, se_ratio=0.25, use_dfc=use_dfc)
            )
            c2 = stage2_blocks[-1].out_channels
        self.stage2 = nn.Sequential(*stage2_blocks)
 
        # Etapa 3: 14x14 -> 7x7 (1 stride=2 + 3 stride=1)
        stage3_blocks = [
            GhostBottleneckV2(c2, mid_chs=240, out_chs=80, dw_kernel_size=3,
                               stride=2, se_ratio=0.0, use_dfc=use_dfc)
        ]
        c3 = stage3_blocks[-1].out_channels
        mids3 = [200, 184, 184]
        se3 = [0.0, 0.0, 0.25]
        for mid, se in zip(mids3, se3):
            stage3_blocks.append(
                GhostBottleneckV2(c3, mid_chs=mid, out_chs=80, dw_kernel_size=3,
                                   stride=1, se_ratio=se, use_dfc=use_dfc)
            )
            c3 = stage3_blocks[-1].out_channels
        self.stage3 = nn.Sequential(*stage3_blocks)
 
        # Etapa 4: 7x7 -> 7x7 (2 bloques stride=1)
        stage4_blocks = []
        c4 = c3
        for mid in [480, 672]:
            stage4_blocks.append(
                GhostBottleneckV2(c4, mid_chs=mid, out_chs=112, dw_kernel_size=3,
                                   stride=1, se_ratio=0.25, use_dfc=use_dfc)
            )
            c4 = stage4_blocks[-1].out_channels
        self.stage4 = nn.Sequential(*stage4_blocks)
 
        # Conv 1x1 expansión
        self.conv_expand = ConvBNPReLU(c4, 256, 1)
 
        # GDC: Global Depthwise Conv 7x7 -> reduce a (1,1)
        self.gdc = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=7, groups=256, bias=False),
            nn.GroupNorm(num_groups=16, num_channels=256)
        )
 
        # Embedding 128-D
        self.embedding = nn.Sequential(
            nn.Conv2d(256, embedding_dim, 1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=embedding_dim),
            nn.Flatten()
        )
 
        # Dropout antes del classifier (regularización contra overfitting)
        self.dropout = nn.Dropout(p=dropout_p)
 
        self.classifier = nn.Linear(embedding_dim, num_classes)
 
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv_expand(x)
        x = self.gdc(x)
        x = self.embedding(x)
        x = self.dropout(x)
        return self.classifier(x)
 
 
model = GhostFaceNetV2(num_classes=NUM_CLASSES, use_dfc=True).to(DEVICE)
print(model)
 
dummy = torch.zeros(1, 1, IMG_HEIGHT, IMG_WIDTH).to(DEVICE)
out   = model(dummy)
print(f"\nShape de salida con input (1,1,112,112): {out.shape}")  

GhostFaceNetV2(
  (stem): ConvBNPReLU(
    (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): PReLU(num_parameters=16)
  )
  (stage1): Sequential(
    (0): GhostBottleneckV2(
      (dfc): DFCAttention(
        (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
        (conv_h): Conv2d(48, 48, kernel_size=(1, 5), stride=(1, 1), padding=(0, 2), groups=48, bias=False)
        (conv_v): Conv2d(48, 48, kernel_size=(5, 1), stride=(1, 1), padding=(2, 0), groups=48, bias=False)
        (bn): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (ghost1): GhostModule(
        (primary_conv): Sequential(
          (0): Conv2d(16, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): PReLU(num_parameters=24)
        

In [9]:
 
def extract_embedding(model, x):
    """Corre el backbone hasta el embedding 128-D normalizado,
    saltando el classifier/dropout (que no se usan con ArcFace)."""
    x = model.stem(x)
    x = model.stage1(x)
    x = model.stage2(x)
    x = model.stage3(x)
    x = model.stage4(x)
    x = model.conv_expand(x)
    x = model.gdc(x)
    x = model.embedding(x)
    x = F.normalize(x, p=2, dim=1)
    return x

In [10]:
def extraer_embeddings(model, imagenes):
    model.eval()
    embeddings = []

    with torch.no_grad():
        for img in imagenes:
            x = img.unsqueeze(0).to(DEVICE)
            emb = extract_embedding(model, x)  
            embeddings.append(emb.squeeze(0).cpu().numpy())

    return np.array(embeddings)   # (N, 128)


def cargar_modelo_ghost(ckpt_path, use_dfc, es_arcface):
    model = GhostFaceNetV2(num_classes=NUM_CLASSES, use_dfc=use_dfc).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    if es_arcface:
        model.load_state_dict(ckpt["model"])
    else:
        model.load_state_dict(ckpt)

    return model

In [11]:
def generar_pares(embeddings, etiquetas, n_impostores=10000, seed=42):
    np.random.seed(seed)
    etiquetas = np.array(etiquetas)
    n = len(embeddings)

    similitudes = []
    labels_pares = []

    for sujeto in np.unique(etiquetas):
        indices = np.where(etiquetas == sujeto)[0]
        for i, j in combinations(indices, 2):
            sim = float(np.dot(embeddings[i], embeddings[j]))
            similitudes.append(sim)
            labels_pares.append(1)

    n_genuinos = len(similitudes)
    print(f"  Pares genuinos   : {n_genuinos}")

    impostores_generados = 0
    intentos = 0
    while impostores_generados < n_impostores and intentos < n_impostores * 10:
        i, j = np.random.randint(0, n, size=2)
        if etiquetas[i] != etiquetas[j]:
            sim = float(np.dot(embeddings[i], embeddings[j]))
            similitudes.append(sim)
            labels_pares.append(0)
            impostores_generados += 1
        intentos += 1

    print(f"  Pares impostores : {impostores_generados}")

    return np.array(similitudes), np.array(labels_pares)

In [12]:
def calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01):
    fpr, tpr, thresholds = roc_curve(labels_pares, similitudes)
    fnr = 1 - tpr

    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)

    indices_far = np.where(fpr <= far_objetivo)[0]
    if len(indices_far) > 0:
        tar = float(tpr[indices_far[-1]])
    else:
        tar = 0.0

    return eer * 100, tar * 100

# Experimentos

In [18]:
experimentos = [
    ("ORL original — CE",         ghost_orl,              ORL_FULL_PATH, False, True),

    ("CLAHE x2 — CE",             ghost_clahe_x2_ce,       CLAHE_FULL_X2, False, True),
    ("CLAHE x2 — ArcFace",        ghost_clahe_x2_arcface,  CLAHE_FULL_X2, True,  True),
    ("CLAHE x2 — sin DFC (CE)",   ghost_clahe_x2_nodfc,    CLAHE_FULL_X2, False, False),

    ("CLAHE x4 — CE",             ghost_clahe_x4_ce,       CLAHE_FULL_X4, False, True),
    ("CLAHE x4 — ArcFace",        ghost_clahe_x4_arcface,  CLAHE_FULL_X4, True,  True),
    ("CLAHE x4 — sin DFC (CE)",   ghost_clahe_x4_nodfc,    CLAHE_FULL_X4, False, False),

    ("SR x2 — CE",                ghost_sr_x2_ce,          SR_FULL_X2,    False, True),
    ("SR x2 — ArcFace",           ghost_sr_x2_arcface,     SR_FULL_X2,    True,  True),
    ("SR x2 — sin DFC (CE)",      ghost_sr_x2_nodfc,       SR_FULL_X2,    False, False),

    ("SR x4 — CE",                ghost_sr_x4_ce,          SR_FULL_X4,    False, True),
    ("SR x4 — ArcFace",           ghost_sr_x4_arcface,     SR_FULL_X4,    True,  True),
    ("SR x4 — sin DFC (CE)",      ghost_sr_x4_nodfc,       SR_FULL_X4,    False, False),

    
    ("Lite-ERSGAN x2 — CE",                ghost_le_x2_ce,          SR_FULL_X2,    False, True),
    ("Lite-ERSGAN x2 — ArcFace",           ghost_le_x2_arcface,     SR_FULL_X2,    True,  True),
    ("Lite-ERSGAN x2 — sin DFC (CE)",      ghost_le_x2_nodfc,       SR_FULL_X2,    False, False),

    ("Lite-ERSGAN x4 — CE",                ghost_le_x4_ce,          SR_FULL_X4,    False, True),
    ("Lite-ERSGAN x4 — ArcFace",           ghost_le_x4_arcface,     SR_FULL_X4,    True,  True),
    ("Lite-ERSGAN x4 — sin DFC (CE)",      ghost_le_x4_nodfc,       SR_FULL_X4,    False, False),
]

resultados_eer_tar = []

for nombre, ckpt_path, dataset_path, es_arcface, use_dfc in experimentos:
    print(f"\n{'='*60}")
    print(f"Evaluando: {nombre}")
    print(f"  Checkpoint : {ckpt_path}")
    print(f"  Dataset    : {dataset_path}")
    print(f"  ArcFace    : {es_arcface} | use_dfc: {use_dfc}")

    try:
        model = cargar_modelo_ghost(ckpt_path, use_dfc=use_dfc, es_arcface=es_arcface)
    except FileNotFoundError:
        print(f"  [SALTADO] Checkpoint no encontrado todavía: {ckpt_path}")
        continue

    imagenes, etiquetas, clases = cargar_dataset_completo(dataset_path)
    print(f"  Imágenes cargadas: {len(imagenes)} ({len(clases)} sujetos)")

    embeddings = extraer_embeddings(model, imagenes)
    print(f"  Embeddings shape : {embeddings.shape}")

    similitudes, labels_pares = generar_pares(embeddings, etiquetas, n_impostores=10000)

    eer, tar_1 = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01)
    _, tar_01  = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.001)

    print(f"  EER              : {eer:.2f}%")
    print(f"  TAR @ FAR=1%     : {tar_1:.2f}%")
    print(f"  TAR @ FAR=0.1%   : {tar_01:.2f}%")

    resultados_eer_tar.append({
        "Experimento": nombre,
        "EER (%)": round(eer, 2),
        "TAR @ FAR=1% (%)": round(tar_1, 2),
        "TAR @ FAR=0.1% (%)": round(tar_01, 2),
    })

df_resultados = pd.DataFrame(resultados_eer_tar)
print("\n" + "="*60)
print("TABLA RESUMEN — EER y TAR (GhostFaceNetV2)")
print("="*60)
print(df_resultados.to_string(index=False))


Evaluando: ORL original — CE
  Checkpoint : ../../Semana 10/GhostFaceNetv2/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_orl.pth
  Dataset    : ../../dataset/ORL_limpio/Training
  ArcFace    : False | use_dfc: True
  [SALTADO] Checkpoint no encontrado todavía: ../../Semana 10/GhostFaceNetv2/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_orl.pth

Evaluando: CLAHE x2 — CE
  Checkpoint : ../../Semana 10/GhostFaceNetv2/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_clahe_x2_experimento2.pth
  Dataset    : ../../dataset/Tecnica De mejora de imagen/ORL_CLAHE_x2/Training
  ArcFace    : False | use_dfc: True
  Imágenes cargadas: 360 (40 sujetos)
  Embeddings shape : (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER              : 1.10%
  TAR @ FAR=1%     : 98.75%
  TAR @ FAR=0.1%   : 93.40%

Evaluando: CLAHE x2 — ArcFace
  Checkpoint : ../../Semana 10/GhostFaceNetv2/GhostFaceNetV2_Modelo base_Clahe x2/best_ghostfacenet_arcface_clahe_x2_experimento3.pt

In [19]:
rank1_conocido = {
    "ORL original — CE":        95.00,

    "CLAHE x2 — CE":             95.00,
    "CLAHE x2 — ArcFace":        95.00,
    "CLAHE x2 — sin DFC (CE)":   92.50,

    "CLAHE x4 — CE":             85.00,
    "CLAHE x4 — ArcFace":        87.50,
    "CLAHE x4 — sin DFC (CE)":   82.50,

    "SR x2 — CE":             95.00,
    "SR x2 — ArcFace":        95.00,
    "SR x2 — sin DFC (CE)":   95.00,

    "SR x4 — CE":             87.50,
    "SR x4 — ArcFace":        92.50,
    "SR x4 — sin DFC (CE)":   87.50,

    "Lite-ERSGAN x2 — CE":             95.00,
    "Lite-ERSGAN x2 — ArcFace":        97.50,
    "Lite-ERSGAN x2 — sin DFC (CE)":   95.00,

    "Lite-ERSGAN x4 — CE":             95.00,
    "Lite-ERSGAN x4 — ArcFace":        97.50,
    "Lite-ERSGAN x4 — sin DFC (CE)":   95.00,   
}

df_resultados["Rank-1 (%)"] = df_resultados["Experimento"].map(rank1_conocido)

df_final = df_resultados[
    ["Experimento", "Rank-1 (%)", "EER (%)", "TAR @ FAR=1% (%)", "TAR @ FAR=0.1% (%)"]
]

print("\nTABLA FINAL — GhostFaceNetV2 (Rank-1 + EER + TAR)")
print("=" * 80)
print(df_final.to_string(index=False))

df_final.to_csv("resultados_eer_tar_ghostfacenetv2.csv", index=False)
print("\nGuardado en: resultados_eer_tar_ghostfacenetv2.csv")


TABLA FINAL — GhostFaceNetV2 (Rank-1 + EER + TAR)
                  Experimento  Rank-1 (%)  EER (%)  TAR @ FAR=1% (%)  TAR @ FAR=0.1% (%)
                CLAHE x2 — CE        95.0     1.10             98.75               93.40
           CLAHE x2 — ArcFace        95.0     0.00            100.00              100.00
      CLAHE x2 — sin DFC (CE)        92.5     0.92             99.17               95.97
                CLAHE x4 — CE        85.0     3.12             92.08               65.21
           CLAHE x4 — ArcFace        87.5     0.21             99.93               99.44
      CLAHE x4 — sin DFC (CE)        82.5     2.31             96.11               78.33
                   SR x2 — CE        95.0     1.74             96.25               79.58
              SR x2 — ArcFace        95.0     0.01            100.00              100.00
         SR x2 — sin DFC (CE)        95.0     3.76             90.14               72.85
                   SR x4 — CE        87.5     1.17         